# What Closes the Generalization Gap in Small-Scale Deep RL?

### What this notebook is
In reinforcement learning, an agent can get very good at the exact situations it
trained on yet fail completely on new ones. The difference between *training*
performance and *held-out* performance is the **generalization gap**. This
notebook measures that gap on a procedurally-generated gridworld and asks a
practical question: **which common tricks actually shrink the gap, and which only
look like they do once you use proper statistics?**

### The experimental logic
We fix a **baseline** agent and then change **one thing at a time** from it — the
number of distinct training levels, the data augmentation, the regularizer, the
network width. Each variant is trained from scratch on several random seeds under
the *same* compute budget, then evaluated on a set of **held-out levels it never
saw**. Changing one variable at a time is what lets us attribute any change in the
gap to that variable.

### What it produces
A tidy `aggregated.csv`, a `SUMMARY.md` you can read top-to-bottom, and three
figures: a success-rate chart, a return-IQM chart (both with confidence
intervals), and learning curves. The console also prints a live readout as it runs.

### How to run it
- One self-contained notebook. The **only** thing you normally change is
  `SMOKE_TEST` in the config cell.
- `SMOKE_TEST = True` → a tiny sweep that finishes in ~1–2 min. It checks that the
  whole pipeline works; the *numbers are not meaningful* at that scale.
- `SMOKE_TEST = False` → the real study. **Use a GPU runtime** (Settings →
  Accelerator → GPU). Everything is end-to-end **JAX**, so all seeds for a
  condition train in parallel (`vmap`) and the full sweep is well within a Kaggle
  session.

### What's worth reading to actually learn the RL
The **PPO cell** (section 3) is the real algorithm — worth reading line by line.
The **metrics cell** (section 4) explains why we use IQM + bootstrap CIs + a
success rate instead of a mean over a few seeds. Those two are the parts to own.

In [ ]:
# Dependencies. We install flax + optax (lightweight) and only install JAX if it
# is missing — Kaggle's GPU image usually ships a GPU-enabled JAX already, and we
# don't want to clobber it with a CPU build. (numpy/pandas/matplotlib are preinstalled.)
# NOTE: we deliberately do NOT use the `rliable` package — its `arch` dependency has
# fragile binary-compat issues; the IQM / bootstrap-CI / probability-of-improvement
# metrics from Agarwal et al. (2021) are implemented directly below.
import os, sys, subprocess, importlib.util
if not os.environ.get("NB_FAST"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flax", "optax"])
    if importlib.util.find_spec("jax") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"])

## 1 · Configuration

This is the only cell you normally touch.

- **`SMOKE_TEST = True`** runs a tiny version (small grid, few levels, 2 seeds, a
  handful of updates) that finishes in ~1–2 minutes. Use it to confirm everything
  runs. The numbers will look degenerate — that's expected.
- **`SMOKE_TEST = False`** runs the real study: a 9×9 grid, a 256-level training
  pool with 100 held-out test levels, 8 seeds per condition, and the full
  intervention sweep, under a fixed per-run step budget.

The cell prints a banner telling you which mode you're in and — importantly —
**which device JAX is using**. If you're on a GPU runtime but it prints `cpu`,
the run will be very slow; install a CUDA build of JAX and restart (the banner
tells you the exact command). The `CHUNKS`/`UPDATES` numbers set the training
budget per run; if the learning-curve figure later shows curves still rising,
raise `UPDATES`.

In [ ]:
SMOKE_TEST = False
SEED0 = 0
OUTDIR = "results_rl"

if SMOKE_TEST:
    GRID, POOL, N_TEST = 6, 24, 8
    BASE_NTRAIN = 4
    PPO_BASE = dict(num_envs=32, num_steps=12, gamma=0.99, gae_lambda=0.95,
                    clip=0.2, ent_coef=0.01, vf_coef=0.5, update_epochs=2, num_minibatches=4, lr=3e-3, dropout=0.0, reg="none",
                    weight_decay=0.0, aug="none", width=1, n_train=BASE_NTRAIN)
    MAX_STEPS, SEEDS, CHUNKS, UPDATES = 36, 2, 3, 8
else:
    GRID, POOL, N_TEST = 9, 356, 100      # 256 train pool + 100 held-out test
    BASE_NTRAIN = 64
    PPO_BASE = dict(num_envs=256, num_steps=64, gamma=0.99, gae_lambda=0.95,
                    clip=0.2, ent_coef=0.01, vf_coef=0.5, update_epochs=4,
                    num_minibatches=8, lr=2.5e-3, dropout=0.0, reg="none",
                    weight_decay=0.0, aug="none", width=1, n_train=BASE_NTRAIN)
    # Budget: in the first full run several test curves were still rising at 30
    # updates, so we extend it. CHUNKS = how many eval points along the curve;
    # UPDATES = PPO updates between eval points. Total = CHUNKS*UPDATES updates
    # per run. If the learning-curve figure still hasn't plateaued, raise UPDATES.
    MAX_STEPS, SEEDS, CHUNKS, UPDATES = 80, 8, 15, 60



# --- hidden fast-path: only used for quick execution checks, ignore on Kaggle ---
import os
if os.environ.get("NB_FAST"):
    GRID, POOL, N_TEST = 5, 12, 4
    BASE_NTRAIN = 3
    PPO_BASE.update(num_envs=8, num_steps=6, update_epochs=1, num_minibatches=2)
    PPO_BASE["n_train"] = BASE_NTRAIN
    MAX_STEPS, SEEDS, CHUNKS, UPDATES = 20, 1, 1, 2

print("=" * 60)
print(f"  MODE: {'SMOKE — quick plumbing test (~1-2 min, numbers NOT meaningful)' if SMOKE_TEST else 'FULL — the real study (GPU, longer)'}")
print(f"  -> set SMOKE_TEST = False above for the real run")
try:
    import jax
    devs = jax.devices()
    print(f"  JAX devices: {devs}")
    if not SMOKE_TEST and devs[0].platform == "cpu":
        print("  WARNING: JAX is on CPU. For the FULL run, use a GPU runtime and, if it")
        print("           still shows cpu, run:  pip install -U 'jax[cuda12]'  then restart.")
except Exception:
    pass
print("=" * 60)

## 2 · Environment — a procedural gridworld in JAX

**What a "level" is.** One level is a single maze: a wall layout, a start cell,
and a goal cell. The agent sees a small 3-channel image (walls / its own position
/ the goal), picks one of 4 moves, gets **+1 for reaching the goal**, a tiny
penalty each step (so faster is better), and the episode ends at the goal or after
a step limit. So a *fully solved* level gives a small positive return, and a level
where the agent never reaches the goal gives a negative return — that sign flip is
what we'll later use to define "solved".

**Why pre-generate levels.** We generate many solvable mazes once in numpy (with a
breadth-first-search check so the goal is always reachable), then hand the arrays
to a pure-JAX environment that just indexes into them. This keeps the environment
fast and `jit`/`vmap`-friendly.

**The train/test split — the heart of the experiment.** Training samples levels
only from the first `n_train` indices; evaluation always uses a **fixed block of
held-out levels at the end** that training never touches. Measuring performance on
those held-out levels is what exposes the generalization gap.

In [ ]:
"""rl_env.py — procedural gridworld in JAX, with precomputed reachable levels.

A "level" is one (wall-map, start, goal) triple. We precompute many levels in
numpy (with a BFS reachability check so every level is solvable), then expose a
pure-JAX env that indexes into those arrays. Training samples levels from the
first `n_train` indices; evaluation runs fixed held-out indices. That split is
what lets us measure a train-vs-test generalization gap.
"""
import numpy as np
import jax, jax.numpy as jnp
from collections import deque
from functools import partial

# ---- level generation (numpy, runs once at setup) -------------------------
def _reachable(walls, start, goal):
    """BFS: is goal reachable from start avoiding walls?"""
    n = walls.shape[0]
    seen = np.zeros_like(walls, dtype=bool)
    q = deque([tuple(start)])
    seen[start[0], start[1]] = True
    while q:
        x, y = q.popleft()
        if (x, y) == tuple(goal):
            return True
        for dx, dy in ((1,0),(-1,0),(0,1),(0,-1)):
            nx, ny = x+dx, y+dy
            if 0 <= nx < n and 0 <= ny < n and not walls[nx,ny] and not seen[nx,ny]:
                seen[nx,ny] = True
                q.append((nx,ny))
    return False

def make_levels(num_levels, grid=7, p_wall=0.18, seed=0):
    """Generate `num_levels` solvable levels. Returns jnp arrays."""
    rng = np.random.default_rng(seed)
    walls_all, starts_all, goals_all = [], [], []
    tries = 0
    while len(walls_all) < num_levels:
        tries += 1
        walls = (rng.random((grid, grid)) < p_wall).astype(np.float32)
        free = np.argwhere(walls == 0)
        if len(free) < 2:
            continue
        idx = rng.choice(len(free), size=2, replace=False)
        start, goal = free[idx[0]], free[idx[1]]
        if _reachable(walls, start, goal):
            walls_all.append(walls); starts_all.append(start); goals_all.append(goal)
    return (jnp.asarray(np.stack(walls_all)),
            jnp.asarray(np.stack(starts_all)).astype(jnp.int32),
            jnp.asarray(np.stack(goals_all)).astype(jnp.int32))

# ---- pure-JAX env ---------------------------------------------------------
# state = (agent_xy[int32 2], t[int32], level_idx[int32], done[bool])
_MOVES = jnp.array([[-1,0],[1,0],[0,-1],[0,1]], dtype=jnp.int32)  # U D L R

def make_env(walls, starts, goals, max_steps=50, step_pen=0.01):
    grid = walls.shape[1]

    def obs_of(agent, level_idx):
        w = walls[level_idx]                       # (grid,grid)
        a = jnp.zeros((grid,grid)).at[agent[0],agent[1]].set(1.0)
        g_xy = goals[level_idx]
        g = jnp.zeros((grid,grid)).at[g_xy[0],g_xy[1]].set(1.0)
        return jnp.stack([w, a, g], axis=-1)       # (grid,grid,3)

    def reset(level_idx):
        agent = starts[level_idx]
        state = (agent, jnp.int32(0), jnp.int32(level_idx), jnp.bool_(False))
        return state, obs_of(agent, level_idx)

    def step(state, action):
        agent, t, level_idx, _ = state
        nxt = agent + _MOVES[action]
        nxt = jnp.clip(nxt, 0, grid-1)
        blocked = walls[level_idx][nxt[0], nxt[1]] > 0.5
        agent2 = jnp.where(blocked, agent, nxt)
        at_goal = jnp.all(agent2 == goals[level_idx])
        t2 = t + 1
        timeout = t2 >= max_steps
        done = at_goal | timeout
        reward = jnp.where(at_goal, 1.0, 0.0) - step_pen
        state2 = (agent2, t2, level_idx, done)
        return state2, obs_of(agent2, level_idx), reward, done, {"at_goal": at_goal}

    return reset, step, obs_of, grid

# ---- sanity test: random policy -------------------------------------------

## 3 · The agent — PPO, PureJaxRL-style

This is the actual reinforcement-learning algorithm, and the cell most worth
reading if you want to *learn* the RL rather than just run it. **PPO (Proximal
Policy Optimization)** is the standard policy-gradient method. In plain terms, the
loop is:

1. **Collect experience.** Run the current policy in many parallel environments
   for a fixed number of steps, recording what it saw, did, and was rewarded.
2. **Estimate how good each action was.** Compute *advantages* with GAE
   (Generalized Advantage Estimation) — roughly, "how much better did this action
   do than expected?"
3. **Improve the policy, but not too fast.** Nudge the policy to make
   good-advantage actions more likely, using PPO's *clipped* objective so no single
   update changes the policy too drastically (that clipping is what makes PPO
   stable). A *value* head is trained alongside to predict expected return, and an
   *entropy* bonus keeps the policy exploring.

The whole loop is written with `jax.lax.scan`, so it compiles to fast device code
and `vmap`s cleanly over seeds. Two functions matter downstream: `train_chunk`
advances every seed by a fixed number of updates (we call it repeatedly to draw
learning curves), and `evaluate` runs the **greedy** (best-action) policy on a
given set of levels so we can score training and held-out levels separately.

**This is also where the interventions live:** network `width`, `dropout`, L2
weight decay (via the optimizer), and observation augmentation (`shift` / `cutout`)
are all controlled by the config passed in.

In [ ]:
"""rl_ppo.py — compact PureJaxRL-style PPO for the gridworld.

Design notes for the reader (this is the part worth understanding):
  * The whole training loop is end-to-end JAX so it jits and `vmap`s over seeds.
  * `train_chunk` advances every seed by a fixed number of PPO updates and is
    called repeatedly from Python so we can interleave evaluation and build
    learning curves.
  * `evaluate` runs the greedy policy on a *fixed* set of level indices, so we
    can score train-pool levels and held-out test levels separately -> the gap.
"""
import jax, jax.numpy as jnp
import numpy as np
import flax.linen as nn
import optax
from functools import partial
from typing import Sequence


# --------------------------- network --------------------------------------
class ActorCritic(nn.Module):
    n_actions: int
    width: int = 1
    dropout: float = 0.0

    @nn.compact
    def __call__(self, x, training: bool = False):
        w = self.width
        x = nn.relu(nn.Conv(16 * w, (3, 3), padding="SAME")(x))
        x = nn.relu(nn.Conv(32 * w, (3, 3), padding="SAME")(x))
        x = x.reshape((x.shape[0], -1))
        h = nn.relu(nn.Dense(64 * w)(x))
        if self.dropout > 0:
            h = nn.Dropout(self.dropout, deterministic=not training)(h)
        logits = nn.Dense(self.n_actions)(h)
        value = nn.Dense(1)(h)[:, 0]
        return logits, value


# ----------------------- small functional helpers -------------------------
def categorical_sample(key, logits):
    return jax.random.categorical(key, logits)

def log_prob(logits, actions):
    logp = jax.nn.log_softmax(logits)
    return jnp.take_along_axis(logp, actions[:, None], axis=-1)[:, 0]

def entropy(logits):
    p = jax.nn.softmax(logits)
    logp = jax.nn.log_softmax(logits)
    return -jnp.sum(p * logp, axis=-1)


def make_ppo(cfg, env, levels):
    """Returns (init_runner, train_chunk, evaluate). `env`=(reset,step,obs_of,grid)."""
    reset, step, obs_of, grid = env
    walls, starts, goals = levels
    n_actions = 4
    net = ActorCritic(n_actions, width=cfg["width"], dropout=cfg["dropout"])

    NUM_ENVS = cfg["num_envs"]
    NUM_STEPS = cfg["num_steps"]
    n_train = cfg["n_train"]
    GAMMA, LAM = cfg["gamma"], cfg["gae_lambda"]
    CLIP, ENT, VF = cfg["clip"], cfg["ent_coef"], cfg["vf_coef"]
    EPOCHS, NMB = cfg["update_epochs"], cfg["num_minibatches"]
    BATCH = NUM_ENVS * NUM_STEPS
    MB = BATCH // NMB
    aug = cfg["aug"]                       # "none" | "shift" | "cutout"

    # vectorized env ops over NUM_ENVS
    v_reset = jax.vmap(reset)
    v_step = jax.vmap(step)
    v_obs = jax.vmap(obs_of)

    def sample_train_levels(key, n):
        return jax.random.randint(key, (n,), 0, n_train)

    def augment(obs, key):
        if aug == "none":
            return obs
        if aug == "shift":
            # random integer shift by up to 1 cell (pad-and-crop), per sample
            pad = jnp.pad(obs, ((0,0),(1,1),(1,1),(0,0)))
            B = obs.shape[0]
            ks = jax.random.randint(key, (B, 2), 0, 3)   # 0..2 -> shift -1..+1
            def crop(o, k):
                return jax.lax.dynamic_slice(o, (k[0], k[1], 0), (grid, grid, obs.shape[-1]))
            return jax.vmap(crop)(pad, ks)
        if aug == "cutout":
            B = obs.shape[0]
            kx = jax.random.randint(key, (B,), 0, grid)
            ky = jax.random.randint(key, (B,), 0, grid)
            xs = jnp.arange(grid)
            def cut(o, cx, cy):
                m = ~((jnp.abs(xs[:,None]-cx) <= 0) & (jnp.abs(xs[None,:]-cy) <= 0))
                return o * m[:, :, None]
            return jax.vmap(cut)(obs, kx, ky)
        return obs

    # ---------------- init ----------------
    def init_runner(key):
        key, kp, kd, kr = jax.random.split(key, 4)
        lvl0 = sample_train_levels(kr, NUM_ENVS)
        states, obs = v_reset(lvl0)
        params = net.init({"params": kp, "dropout": kd}, obs, training=False)
        opt = make_opt()
        opt_state = opt.init(params)
        return dict(params=params, opt_state=opt_state, states=states, obs=obs, key=key)

    def make_opt():
        chain = [optax.clip_by_global_norm(0.5)]
        if cfg["reg"] == "l2":
            chain.append(optax.add_decayed_weights(cfg["weight_decay"]))
        chain.append(optax.adam(cfg["lr"]))
        return optax.chain(*chain)
    opt = make_opt()

    # ---------------- one PPO update ----------------
    def _update(runner):
        params, opt_state = runner["params"], runner["opt_state"]
        states, obs, key = runner["states"], runner["obs"], runner["key"]

        # ---- collect a rollout of NUM_STEPS ----
        def env_step(carry, _):
            states, obs, key = carry
            key, ka, kr = jax.random.split(key, 3)
            logits, value = net.apply(params, obs, training=False)
            actions = categorical_sample(ka, logits)
            lp = log_prob(logits, actions)
            states2, obs2, reward, done, info = v_step(states, actions)
            # autoreset finished envs to freshly sampled training levels
            new_lvl = sample_train_levels(kr, NUM_ENVS)
            rstates, robs = v_reset(new_lvl)
            states2 = jax.tree_util.tree_map(lambda a, b: jnp.where(
                done.reshape((-1,) + (1,)*(a.ndim-1)), b, a), states2, rstates)
            obs2 = jnp.where(done[:, None, None, None], robs, obs2)
            trans = (obs, actions, lp, value, reward, done)
            return (states2, obs2, key), trans

        (states, obs, key), traj = jax.lax.scan(
            env_step, (states, obs, key), None, length=NUM_STEPS)
        _, last_val = net.apply(params, obs, training=False)

        # ---- GAE ----
        obs_b, act_b, lp_b, val_b, rew_b, done_b = traj
        def gae_step(carry, x):
            gae, next_val = carry
            val, rew, done = x
            delta = rew + GAMMA * next_val * (1 - done) - val
            gae = delta + GAMMA * LAM * (1 - done) * gae
            return (gae, val), gae
        _, adv = jax.lax.scan(
            gae_step, (jnp.zeros(NUM_ENVS), last_val),
            (val_b, rew_b, done_b), reverse=True)
        ret_b = adv + val_b

        # flatten (NUM_STEPS, NUM_ENVS, ...) -> (BATCH, ...)
        def flat(x): return x.reshape((BATCH,) + x.shape[2:])
        data = dict(obs=flat(obs_b), act=flat(act_b), lp=flat(lp_b),
                    val=flat(val_b), adv=flat(adv), ret=flat(ret_b))

        # ---- PPO epochs ----
        def epoch(carry, _):
            params, opt_state, key = carry
            key, kperm, kaug = jax.random.split(key, 3)
            perm = jax.random.permutation(kperm, BATCH)
            d = jax.tree_util.tree_map(lambda x: x[perm], data)

            def minibatch(carry, idx):
                params, opt_state, kaug = carry
                kaug, ka = jax.random.split(kaug)
                mb = jax.tree_util.tree_map(
                    lambda x: jax.lax.dynamic_slice_in_dim(x, idx*MB, MB, 0), d)

                def loss_fn(params, ka):
                    o = augment(mb["obs"], ka)
                    logits, value = net.apply(
                        params, o, training=True,
                        rngs={"dropout": ka})
                    newlp = log_prob(logits, mb["act"])
                    ratio = jnp.exp(newlp - mb["lp"])
                    A = (mb["adv"] - mb["adv"].mean()) / (mb["adv"].std() + 1e-8)
                    l1 = ratio * A
                    l2 = jnp.clip(ratio, 1-CLIP, 1+CLIP) * A
                    pg = -jnp.minimum(l1, l2).mean()
                    v_clip = mb["val"] + jnp.clip(value - mb["val"], -CLIP, CLIP)
                    vl = 0.5 * jnp.maximum((value-mb["ret"])**2,
                                           (v_clip-mb["ret"])**2).mean()
                    ent = entropy(logits).mean()
                    return pg + VF*vl - ENT*ent, (pg, vl, ent)

                (loss, aux), grads = jax.value_and_grad(loss_fn, has_aux=True)(params, ka)
                updates, opt_state = opt.update(grads, opt_state, params)
                params = optax.apply_updates(params, updates)
                return (params, opt_state, kaug), loss

            (params, opt_state, _), losses = jax.lax.scan(
                minibatch, (params, opt_state, kaug), jnp.arange(NMB))
            return (params, opt_state, key), losses.mean()

        (params, opt_state, key), _ = jax.lax.scan(
            epoch, (params, opt_state, key), None, length=EPOCHS)

        mean_rollout_return = rew_b.sum(axis=0).mean()  # rough train signal
        runner = dict(params=params, opt_state=opt_state, states=states, obs=obs, key=key)
        return runner, mean_rollout_return

    @partial(jax.jit, static_argnums=(1,))
    def train_chunk(runner, n_updates):
        def body(runner, _):
            return _update(runner)
        runner, rets = jax.lax.scan(body, runner, None, length=n_updates)
        return runner, rets.mean()

    # ---------------- evaluation on fixed levels ----------------
    def evaluate(params, level_indices, max_eval_steps):
        """Greedy policy return averaged over the given fixed levels."""
        def run_level(lvl):
            state, obs = reset(lvl)
            def body(carry):
                state, obs, ret, done, t = carry
                logits, _ = net.apply(params, obs[None], training=False)
                a = jnp.argmax(logits[0])
                state2, obs2, r, d, info = step(state, a)
                return (state2, obs2, ret + r * (1-done), done | d, t+1)
            def cond(carry):
                *_, done, t = carry
                return (~done) & (t < max_eval_steps)
            init = (state, obs, jnp.float32(0.0), jnp.bool_(False), jnp.int32(0))
            state, obs, ret, done, t = jax.lax.while_loop(cond, body, init)
            return ret
        return jax.vmap(run_level)(level_indices)   # per-level returns (vector)

    return init_runner, train_chunk, evaluate


# --------------------------- smoke training -------------------------------

## 4 · Metrics, the intervention sweep & analysis

**Why not just average a few seeds?** RL results are noisy, and the field has a
history of "improvements" that vanish under careful evaluation. We follow Agarwal
et al. 2021 (*Deep RL at the Edge of the Statistical Precipice*) and report three
complementary things, all implemented directly here (no fragile dependency):

- **Success rate** — the fraction of held-out levels the greedy agent actually
  solves (reaches the goal). This is the headline metric because it is intuitive
  and, crucially, it **does not saturate**: even when every agent's *return* is
  pinned at the timeout floor, success rate can still tell conditions apart.
- **IQM (interquartile mean)** of held-out return — drops the top and bottom 25%
  of scores, so it's far less noisy than the mean and less pessimistic than the
  median. Note it *does* floor at the timeout penalty when most levels are unsolved.
- **Bootstrap confidence intervals** and **probability of improvement vs the
  baseline**, so every claim comes with an honest interval, not a point estimate.

**What gets built:** `build_conditions` defines the one-at-a-time sweep;
`run_condition` trains all seeds for a condition and returns per-level train/test
returns; `analyze_and_save` writes the tidy CSV, **three figures** (success rate,
return-IQM, learning curves), and a `SUMMARY.md` that includes a "how to read
this" note and a comparison of the mean vs the robust picture (the heart of RQ4 —
*do conclusions change under proper statistics?*).

In [ ]:
"""rl_experiment.py — run the intervention sweep and analyze with rliable."""
import os, json, time
import numpy as np, pandas as pd
import jax, jax.numpy as jnp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- rliable metrics (Agarwal et al. 2021), implemented directly for robustness ---
def iqm(x):
    """Interquartile mean: mean of the middle 50% of all scores."""
    v = np.sort(np.asarray(x).ravel())
    n = len(v); lo, hi = int(0.25 * n), int(np.ceil(0.75 * n))
    return float(v[lo:hi].mean()) if hi > lo else float(v.mean())

def bootstrap_ci(x, stat=iqm, reps=2000, alpha=0.05, seed=0):
    """Run-level (seed) bootstrap CI for an aggregate statistic."""
    x = np.asarray(x); rng = np.random.default_rng(seed); n = x.shape[0]
    boots = [stat(x[rng.integers(0, n, n)]) for _ in range(reps)]
    return float(np.percentile(boots, 100*alpha/2)), float(np.percentile(boots, 100*(1-alpha/2)))

def prob_improvement(X, Y):
    """P(X > Y): common-language effect size over all score pairs (ties=0.5)."""
    a, b = np.asarray(X).ravel(), np.asarray(Y).ravel()
    gt = (a[:, None] > b[None, :]).mean()
    eq = (a[:, None] == b[None, :]).mean()
    return float(gt + 0.5 * eq)


def build_conditions(smoke):
    """The one-at-a-time intervention sweep from a fixed baseline."""
    base = dict(n_train=BASE_NTRAIN, aug="none", reg="none",
                weight_decay=0.0, dropout=0.0, width=1)
    conds = [("baseline", "baseline", "-", base)]
    if smoke:
        conds += [
            ("levels",   "n_train", "16",   {**base, "n_train": 16}),
            ("aug",      "shift",   "shift",{**base, "aug": "shift"}),
            ("reg",      "l2",      "l2",   {**base, "reg": "l2", "weight_decay": 1e-4}),
            ("width",    "width",   "x2",   {**base, "width": 2}),
        ]
    else:
        # denser sampling through the 64->256 transition, where generalization emerged
        for nl in [2, 4, 16, 64, 96, 128, 192, 256]:
            conds.append(("levels", "n_train", str(nl), {**base, "n_train": nl}))
        for a in ["shift", "cutout"]:
            conds.append(("aug", "augment", a, {**base, "aug": a}))
        conds.append(("reg", "l2", "l2", {**base, "reg": "l2", "weight_decay": 1e-4}))
        conds.append(("reg", "dropout", "0.1", {**base, "dropout": 0.1}))
        conds.append(("width", "width", "x2", {**base, "width": 2}))
    return conds


def run_condition(cfg_over, env, levels, test_levels, max_steps, seeds, chunks, updates):
    cfg = {**PPO_BASE, **cfg_over}
    init_runner, train_chunk, evaluate = make_ppo(cfg, env, levels)
    keys = jax.random.split(jax.random.PRNGKey(SEED0), seeds)
    runners = jax.vmap(init_runner)(keys)
    v_chunk = jax.vmap(lambda r: train_chunk(r, updates))
    n_tr_eval = min(cfg["n_train"], test_levels.shape[0])
    tr_levels = jnp.arange(n_tr_eval)
    v_eval_tr = jax.vmap(lambda p: evaluate(p, tr_levels, max_steps))     # (seeds, n_tr_eval)
    v_eval_te = jax.vmap(lambda p: evaluate(p, test_levels, max_steps))   # (seeds, n_test)
    te_curve = []
    for _ in range(chunks):
        runners, _ = v_chunk(runners)
        te_curve.append(np.asarray(v_eval_te(runners["params"])).mean(axis=1))  # (seeds,)
    train_mat = np.asarray(v_eval_tr(runners["params"]))   # (seeds, n_tr_eval)
    test_mat = np.asarray(v_eval_te(runners["params"]))    # (seeds, n_test)
    te_curve = np.stack(te_curve, axis=1)                  # (seeds, chunks)
    return train_mat, test_mat, te_curve


def analyze_and_save(records, outdir):
    os.makedirs(f"{outdir}/figures", exist_ok=True)
    SOLVED = 0.0  # a level is "solved" if greedy return > 0 (goal reached; timeouts are negative)

    # ---- tidy aggregated table (per condition x seed) ----
    # We log BOTH the mean return and the success rate (fraction of levels solved).
    # Success rate is the metric that does NOT saturate at the timeout floor, so it
    # can separate conditions that the return-IQM lumps together.
    rows = []
    for r in records:
        seeds = r["test_mat"].shape[0]
        for s in range(seeds):
            tr  = float(r["train_mat"][s].mean())
            te  = float(r["test_mat"][s].mean())
            trs = float((r["train_mat"][s] > SOLVED).mean())   # train success rate
            tes = float((r["test_mat"][s]  > SOLVED).mean())   # test  success rate
            rows.append(dict(condition=r["name"], intervention=r["intervention"],
                             setting=r["setting"], seed=s,
                             final_train_return=tr, final_test_return=te,
                             final_train_success=trs, final_test_success=tes,
                             gap=tr - te, success_gap=trs - tes,
                             sample_efficiency_auc=float(r["te_curve"][s].mean())))
    df = pd.DataFrame(rows)
    df.to_csv(f"{outdir}/aggregated.csv", index=False)

    # ---- per-condition aggregates ----
    labels = [f"{r['name']}|{r['setting']}" for r in records]
    score_dict = {f"{r['name']}|{r['setting']}": r["test_mat"] for r in records}            # returns matrix
    succ_dict  = {f"{r['name']}|{r['setting']}": (r["test_mat"] > SOLVED).mean(axis=1) for r in records}  # per-seed solved-fraction
    iqm_scores = {k: iqm(v) for k, v in score_dict.items()}
    iqm_cis    = {k: bootstrap_ci(v, iqm) for k, v in score_dict.items()}
    succ_mean  = {k: float(np.mean(v)) for k, v in succ_dict.items()}
    succ_cis   = {k: bootstrap_ci(v, np.mean) for k, v in succ_dict.items()}   # CI of the rate across seeds

    base_key   = [k for k in score_dict if k.startswith("baseline")][0]
    poi        = {k: prob_improvement(score_dict[k], score_dict[base_key]) for k in score_dict if k != base_key}

    # ---- figure 1: test-return IQM with CIs (note: this floors at the timeout penalty) ----
    centers = [iqm_scores[k] for k in labels]
    lo = [iqm_cis[k][0] for k in labels]; hi = [iqm_cis[k][1] for k in labels]
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(labels) + 1.5))
    y = np.arange(len(labels))
    ax.errorbar(centers, y, xerr=[np.array(centers)-np.array(lo), np.array(hi)-np.array(centers)],
                fmt="o", capsize=4, color="#0E7C7B")
    ax.set_yticks(y); ax.set_yticklabels(labels); ax.invert_yaxis()
    ax.set_xlabel("Test-level return  (IQM, 95% CI)"); ax.set_title("Generalization by intervention — return")
    ax.grid(axis="x", alpha=0.3); fig.tight_layout()
    fig.savefig(f"{outdir}/figures/test_return_iqm.png", dpi=130); plt.close(fig)

    # ---- figure 2: test SUCCESS RATE with CIs (the non-saturating headline metric) ----
    centers = [succ_mean[k] for k in labels]
    lo = [succ_cis[k][0] for k in labels]; hi = [succ_cis[k][1] for k in labels]
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(labels) + 1.5))
    ax.barh(y, centers, color="#0E7C7B", alpha=0.45)
    ax.errorbar(centers, y, xerr=[np.array(centers)-np.array(lo), np.array(hi)-np.array(centers)],
                fmt="o", capsize=4, color="#0A5C5B")
    ax.axvline(succ_mean[base_key], ls="--", color="#B4612F", lw=1.2, label="baseline")
    ax.set_yticks(y); ax.set_yticklabels(labels); ax.invert_yaxis()
    ax.set_xlabel("Fraction of held-out levels solved  (mean, 95% CI)"); ax.set_xlim(0, 1)
    ax.set_title("Generalization by intervention — success rate")
    ax.legend(fontsize=8); ax.grid(axis="x", alpha=0.3); fig.tight_layout()
    fig.savefig(f"{outdir}/figures/test_success_rate.png", dpi=130); plt.close(fig)

    # ---- figure 3: learning curves (are the test curves still rising? = budget check) ----
    fig, ax = plt.subplots(figsize=(7.5, 4.7))
    for r in records:
        m = r["te_curve"].mean(axis=0)
        ax.plot(np.arange(1, len(m)+1), m, marker="o", ms=3, label=f"{r['name']}|{r['setting']}")
    ax.set_xlabel("eval point (training progress)"); ax.set_ylabel("test return (mean over seeds)")
    ax.set_title("Test-return learning curves  (still rising => budget too short)")
    ax.legend(fontsize=6, ncol=2); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(f"{outdir}/figures/learning_curves.png", dpi=130); plt.close(fig)

    # ---- SUMMARY.md (with a short 'how to read' so the file is self-explanatory) ----
    g = df.groupby(["condition", "setting"]).agg(
        test=("final_test_return", "mean"), train=("final_train_return", "mean"),
        solved=("final_test_success", "mean"), gap=("gap", "mean")).reset_index()
    order = sorted(labels, key=lambda k: succ_mean[k], reverse=True)
    lines = [
        "# RL Generalization-Gap — Results Summary", "",
        f"_Generated {time.strftime('%Y-%m-%d %H:%M')} · {'SMOKE' if SMOKE_TEST else 'FULL'} run · "
        f"{records[0]['test_mat'].shape[0]} seeds_", "",
        "**How to read this.** Every agent trains to near-optimal on its *training* levels; "
        "the question is how it does on *held-out* levels. `Solved` = fraction of held-out "
        "levels the greedy policy reaches the goal on (this metric does not saturate). "
        "`Test IQM` = interquartile mean of held-out return (robust, but floors at the "
        "timeout penalty when most levels are unsolved). `Gap` = train − test return. "
        "`P(improve)` = probability a random (seed, level) beats the baseline.", "",
        "## Ranked by held-out success rate", "",
        "| Condition | Solved | 95% CI | Test IQM | P(improve) |", "|---|---|---|---|---|"]
    for k in order:
        p = "—" if k == base_key else f"{poi.get(k, float('nan')):.2f}"
        lines.append(f"| {k} | {succ_mean[k]*100:.0f}% | "
                     f"[{succ_cis[k][0]*100:.0f}%, {succ_cis[k][1]*100:.0f}%] | "
                     f"{iqm_scores[k]:+.3f} | {p} |")
    lines += ["", "## Mean train / test / gap (return)", "",
              "| Condition | Setting | Train | Test | Solved | Gap |", "|---|---|---|---|---|---|"]
    for _, r in g.iterrows():
        lines.append(f"| {r['condition']} | {r['setting']} | {r['train']:+.3f} | "
                     f"{r['test']:+.3f} | {r['solved']*100:.0f}% | {r['gap']:+.3f} |")
    with open(f"{outdir}/SUMMARY.md", "w") as f:
        f.write("\n".join(lines))
    return df

## 5 · Run the sweep

This trains every condition and writes the results. **Follow along in the output:**
the cell first prints a summary of exactly what will run (grid size, number of
conditions and seeds, the per-run step budget). Then, as each condition finishes,
it prints a live line — `train`, `test`, **`solved%`** (fraction of held-out
levels solved), `gap`, and the wall-clock time — so you can watch the story emerge
rather than wait for a black box. Finally it prints a **ranked table** of all
conditions by held-out success rate, with each one's improvement over the baseline.

What you should see in a healthy run: `train` near +0.9 for every condition (the
agents master their training levels), while `test` and `solved%` vary a lot across
conditions — that spread *is* the result. Files written: `results_rl/SUMMARY.md`,
`results_rl/aggregated.csv`, and three figures under `results_rl/figures/`.

In [ ]:
t0 = time.time()
os.makedirs(OUTDIR, exist_ok=True)
mode = "SMOKE (quick plumbing test)" if SMOKE_TEST else "FULL (the real study)"
walls, starts, goals = make_levels(POOL, grid=GRID, seed=123)
env = make_env(walls, starts, goals, max_steps=MAX_STEPS)
test_levels = jnp.arange(POOL - N_TEST, POOL)        # the fixed held-out set
conds = build_conditions(SMOKE_TEST)
if os.environ.get('NB_FAST'): conds = conds[:3]
if os.environ.get("NB_FAST"):
    conds = conds[:3]

# ---- print exactly what is about to run, so there are no surprises ----
total_updates = CHUNKS * UPDATES
steps_per_seed = total_updates * PPO_BASE["num_envs"] * PPO_BASE["num_steps"]
steps_str = f"{steps_per_seed/1e6:.1f}M" if steps_per_seed >= 1e6 else f"{steps_per_seed/1e3:.0f}k"
print("=" * 66)
print(f"  {mode}")
print(f"  env          : {GRID}x{GRID} grid | {POOL-N_TEST}-level train pool | {N_TEST} held-out test levels")
print(f"  sweep        : {len(conds)} conditions x {SEEDS} seeds (seeds run in parallel via vmap)")
print(f"  budget/run   : {CHUNKS} eval-points x {UPDATES} updates = {total_updates} PPO updates")
print(f"                 ~{steps_str} environment steps per seed")
print(f"  'solved'     : greedy policy reaches the goal on a held-out level (return > 0)")
print(f"  what to watch: train should hit ~+0.9 everywhere; the story is in TEST + SOLVED%")
print("=" * 66)

records = []
for i, (name, interv, setting, cfg_over) in enumerate(conds, 1):
    tc = time.time()
    train_mat, test_mat, te_curve = run_condition(
        cfg_over, env, (walls, starts, goals), test_levels,
        MAX_STEPS, SEEDS, CHUNKS, UPDATES)
    records.append(dict(name=name, intervention=interv, setting=setting,
                        train_mat=train_mat, test_mat=test_mat, te_curve=te_curve))
    tr, te = train_mat.mean(), test_mat.mean()
    solved = (test_mat > 0).mean()
    tag = "  <- generalizes" if solved > 0.05 else ""
    print(f"  [{i:2d}/{len(conds)}] {name+'|'+setting:14s}  "
          f"train={tr:+.2f}  test={te:+.2f}  solved={solved*100:3.0f}%  "
          f"gap={tr-te:+.2f}   ({time.time()-tc:.0f}s){tag}")

print("\n  analyzing (IQM + bootstrap CIs + success rate) and writing figures ...")
df = analyze_and_save(records, OUTDIR)

# ---- ranked readout: the story in the console, not just in the files ----
agg = (df.groupby(["condition", "setting"])
         .agg(test=("final_test_return", "mean"), solved=("final_test_success", "mean"),
              gap=("gap", "mean")).reset_index()
         .sort_values("solved", ascending=False))
base_solved = float(df[df.condition == "baseline"]["final_test_success"].mean())
print("\n  " + "-" * 62)
print("  RANKED BY HELD-OUT SUCCESS RATE")
print(f"  {'condition':16s} {'solved':>7s} {'test_ret':>9s} {'gap':>7s}   vs baseline")
for _, r in agg.iterrows():
    d = r["solved"] - base_solved
    flag = ("baseline" if r["condition"] == "baseline"
            else f"+{d*100:.0f} pts" if d > 0.01 else (f"{d*100:.0f} pts" if d < -0.01 else "~same"))
    print(f"  {r['condition']+'|'+r['setting']:16s} {r['solved']*100:6.0f}% "
          f"{r['test']:+9.3f} {r['gap']:+7.3f}   {flag}")
print("  " + "-" * 62)
print(f"\n  Wrote {OUTDIR}/SUMMARY.md, aggregated.csv, and 3 figures.  (total {time.time()-t0:.0f}s)")

## 6 · Scaling to the real study, and reading the results

**To run the real study:** set a **GPU** runtime and `SMOKE_TEST = False`. This
uses a 9×9 grid, a 256-level training pool with 100 held-out test levels, 8 seeds,
and the full one-at-a-time sweep: number of training levels
`∈ {2, 4, 16, 64, 96, 128, 192, 256}` (densely sampled so you can see *where*
generalization turns on), augmentation `∈ {shift, cutout}`, regularization
`∈ {L2, dropout}`, and width ×2.

**Reading the output, in order of what matters:**
1. **Success-rate chart + ranked table** — the clearest signal. Which conditions
   solve more held-out levels than baseline, and do their confidence intervals
   clear it?
2. **Learning-curve figure** — if curves are *still rising* at the last eval point,
   the budget is too short and the numbers will move with more training; raise
   `UPDATES` in the config and rerun. (This is a genuine check, not a formality.)
3. **Return-IQM chart** — robust, but expect it to floor at the timeout penalty
   for conditions that solve few levels; that flooring is itself informative.

**Practical tips:** if you hit GPU memory limits, lower `PPO_BASE["num_envs"]` or
`SEEDS`. To validate beyond this gridworld for a paper, swap in **XLand-MiniGrid**
or a **Procgen-easy** subset — the PPO code only depends on the env interface
(`reset`, `step`, `obs_of`).

**Before the definitive run, pre-register:** freeze `build_conditions`, the seed
count, and the step budget in a `PREREG.md`, then report whatever you find — nulls
included. Committing to the analysis before seeing results is what makes the study
trustworthy, and it's the whole point of the paper.

In [ ]:
! git clone https://github.com/Niranjan-GopaL/DeepRL.git

In [ ]:
! git status

In [ ]:
! git config --global user.name "Niranjan-GopaL"

! git config --global user.email "niranjan.gopal@iiitb.ac.in"

! git config --global --list

! git add . && git commit -m "Second Run"

In [ ]:
! git remote set-url origin "https://$(cat ./../secrets.txt)@github.com/Niranjan-GopaL/DeepRL.git"

In [ ]:
! git push